In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/pre_cell_26.pickle

In [ ]:
%%RecordEvent
%%time
### cell 26 ###

# 1. Restrict to first 5 rows (GPU)
train_text_df = train_text_df.head(5)

# 1. Tokenize each essay into one row per token, with token indices
tokenized = (
    train_text_df[['id', 'text']]
    .assign(tokens=train_text_df['text'].str.split(' '))
    .explode('tokens')
    .reset_index(drop=True)
)
tokenized['token_index'] = tokenized.groupby('id').cumcount()
print(1)

# 2. Expand the discourse annotations into one row per token index with B-/I- labels
labels = (
    train[['id', 'discourse_type', 'predictionstring']]
    .reset_index()
    .rename(columns={'index': 'disc_row'})
    # split into string tokens and explode (all on GPU)
    .assign(token_index_str=lambda df: df['predictionstring'].str.split(' '))
    .explode('token_index_str')
    .reset_index(drop=True)
)
# convert token indices to ints, compute position in discourse
labels['token_index'] = labels['token_index_str'].astype('int32')
labels['pos_in_discourse'] = labels.groupby('disc_row').cumcount()
# build B-/I- labels vectorized
labels['label'] = (
    ('B-' + labels['discourse_type'])
    .where(labels['pos_in_discourse'] == 0,
           'I-' + labels['discourse_type'])
)
# keep only needed columns
labels = labels[['id', 'token_index', 'label']]
print(2)

# 3. Merge tokenized text with labels, defaulting missing labels to "O"
tokenized = tokenized.merge(labels, on=['id', 'token_index'], how='left')
tokenized['label'] = tokenized['label'].fillna('O')
print(3)

# 4. Aggregate back to one list of labels per essay (GPU list aggregation)
entities_df = (
    tokenized.groupby('id')['label']
    .agg_list()
    .reset_index()
    .rename(columns={'label': 'entities'})
)
print(4)

# 5. Attach the entities column to the original DataFrame
train_text_df = train_text_df.merge(entities_df, on='id', how='left')
print(5)
print(6)

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_26/checkpoints/post_cell_26_try_8.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_26_try_8.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[26], f)


In [ ]:
opt_output = Out.get(4)